# Task 3 - Data Preprocessing and Feature Engineering

This notebook improves the quality of the dataset for eLCS while avoiding data leakage.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
# Start from the original dataset so Task 3 is reproducible
data = pd.read_csv("../data/patient_data.csv")

print("Dataset shape:", data.shape)
print("Missing values:", data.isna().sum().sum())
print("Exact duplicate rows:", data.duplicated().sum())

Dataset shape: (280985, 39)
Missing values: 0
Exact duplicate rows: 2284


## 3.1 Target and Leakage Prevention


In [3]:
classLabel = "heart_disease_target"

# HT in sublabel represents a record containing heart disease
data[classLabel] = (
    data["sublabel"]
    .str.contains("HT", na=False)
    .astype(int)
)

print(data[classLabel].value_counts())
print()
print(data[classLabel].value_counts(normalize=True))

heart_disease_target
0    275081
1      5904
Name: count, dtype: int64

heart_disease_target
0    0.978988
1    0.021012
Name: proportion, dtype: float64


## 3.2 Exact Duplicate Records


In [4]:
rows_before = len(data)

# Remove complete duplicate records
data_clean = data.drop_duplicates().copy()

rows_after = len(data_clean)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)
print("Remaining exact duplicates:", data_clean.duplicated().sum())

Rows before: 280985
Rows after: 278701
Rows removed: 2284
Remaining exact duplicates: 0


In [5]:
leakage_columns = [
    "sublabel",
    "disease_flags",
    "label",
    "heart_disease"
]

X = data_clean.drop(
    columns=leakage_columns + [classLabel]
)

y = data_clean[classLabel]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nPositive rate:")
print(y.mean())

Predictor shape: (278701, 35)
Target shape: (278701,)

Target distribution:
heart_disease_target
0    272810
1      5891
Name: count, dtype: int64

Positive rate:
0.021137347910484713


In [6]:
# Check whether multiple records now share the same predictor values
duplicate_feature_rows = X.duplicated(
    keep=False
)

print(
    "Rows sharing predictor values:",
    duplicate_feature_rows.sum()
)

print(
    "Unique repeated predictor profiles:",
    X[duplicate_feature_rows]
    .drop_duplicates()
    .shape[0]
)

Rows sharing predictor values: 524
Unique repeated predictor profiles: 262


In [7]:
profile_check = X.copy()
profile_check[classLabel] = y.values

target_counts_per_profile = (
    profile_check
    .groupby(list(X.columns))[classLabel]
    .nunique()
)

conflicting_profiles = target_counts_per_profile[
    target_counts_per_profile > 1
]

print(
    "Predictor profiles with conflicting targets:",
    len(conflicting_profiles)
)

Predictor profiles with conflicting targets: 262


In [8]:
SEED = 3

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining positive rate:", y_train.mean())
print("Testing positive rate:", y_test.mean())

Training shape: (222960, 35)
Testing shape: (55741, 35)

Training positive rate: 0.02113832077502691
Testing positive rate: 0.021133456522129133


## 3.3 Missing Values and Data Types

Missing values and data types are checked again after splitting the cleaned dataset. Any preprocessing rule that must learn information from the data will be fitted using the training set only.

In [9]:
print("Training missing values:", X_train.isna().sum().sum())
print("Testing missing values:", X_test.isna().sum().sum())

print("\nTraining data types:")
print(X_train.dtypes.value_counts())

Training missing values: 0
Testing missing values: 0

Training data types:
float64    18
str        16
int64       1
Name: count, dtype: int64


In [10]:
categorical_columns = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Categorical columns:", len(categorical_columns))

for column in categorical_columns:
    print(f"\n{column}:")
    print(sorted(
        X_train[column]
        .dropna()
        .astype(str)
        .unique()
    ))

Categorical columns: 16

composite_key:
['Mature_Female_Normal_Current_No', 'Mature_Female_Normal_Current_Yes', 'Mature_Female_Normal_Former_No', 'Mature_Female_Normal_Former_Yes', 'Mature_Female_Normal_Never_No', 'Mature_Female_Normal_Never_Yes', 'Mature_Female_Obese_Current_No', 'Mature_Female_Obese_Current_Yes', 'Mature_Female_Obese_Former_No', 'Mature_Female_Obese_Former_Yes', 'Mature_Female_Obese_Never_No', 'Mature_Female_Obese_Never_Yes', 'Mature_Female_Overweight_Current_No', 'Mature_Female_Overweight_Current_Yes', 'Mature_Female_Overweight_Former_No', 'Mature_Female_Overweight_Former_Yes', 'Mature_Female_Overweight_Never_No', 'Mature_Female_Overweight_Never_Yes', 'Mature_Female_Underweight_Current_No', 'Mature_Female_Underweight_Current_Yes', 'Mature_Female_Underweight_Former_No', 'Mature_Female_Underweight_Former_Yes', 'Mature_Female_Underweight_Never_No', 'Mature_Female_Underweight_Never_Yes', 'Mature_Male_Normal_Current_No', 'Mature_Male_Normal_Current_Yes', 'Mature_Male_Nor

In [11]:
numerical_columns = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

numerical_summary = X_train[
    numerical_columns
].describe().T

numerical_summary

,count,mean,std,min,25%,50%,75%,max
age,222960.0,49.725368,21.682929,2.000000,32.000000,50.000000,68.000000,89.000000
age_normalized,222960.0,0.548567,0.249229,0.000000,0.344827,0.551724,0.758620,1.000000
bmi,222960.0,27.490879,7.067417,10.010000,22.000000,27.320000,32.700000,95.690000
hypertension,222960.0,0.129267,0.187690,0.000000,0.000000,0.071823,0.217063,1.000000
HbA1c_level,222960.0,5.952595,0.952922,3.500000,5.387500,5.800000,6.865766,9.000000
glucose,222960.0,135.266042,38.471530,70.000000,100.000000,137.000000,160.000000,300.000000
cholesterol,222960.0,223.985144,35.681088,150.000000,201.000000,225.000000,243.000000,300.000000
sleep_hours,222960.0,6.954918,1.432134,4.000000,6.200000,6.975185,7.700000,10.000000
triglycerides,222960.0,178.468709,67.321256,50.000000,128.000000,177.000000,241.750000,400.000000
blood_pressure,222960.0,149.890027,4.165678,120.000000,148.946429,149.713320,150.720942,180.000000


In [12]:
low_cardinality_numeric = []

for column in numerical_columns:
    unique_count = X_train[column].nunique(dropna=True)

    if unique_count <= 10:
        low_cardinality_numeric.append(column)

print("Low-cardinality numerical columns:")

for column in low_cardinality_numeric:
    print(f"\n{column}:")
    print(sorted(X_train[column].dropna().unique()))

Low-cardinality numerical columns:


In [13]:
range_check = X_train[
    numerical_columns
].agg(["min", "max"]).T

range_check

,min,max
age,2.000000,89.000000
age_normalized,0.000000,1.000000
bmi,10.010000,95.690000
hypertension,0.000000,1.000000
HbA1c_level,3.500000,9.000000
glucose,70.000000,300.000000
cholesterol,150.000000,300.000000
sleep_hours,4.000000,10.000000
triglycerides,50.000000,400.000000
blood_pressure,120.000000,180.000000


In [14]:
negative_value_counts = {}

for column in numerical_columns:
    count = (X_train[column] < 0).sum()

    if count > 0:
        negative_value_counts[column] = count

negative_value_counts

{}

## 3.4 Outlier Investigation



In [15]:
outlier_summary = []

for column in numerical_columns:
    q1 = X_train[column].quantile(0.25)
    q3 = X_train[column].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (X_train[column] < lower_bound)
        | (X_train[column] > upper_bound)
    ).sum()

    outlier_summary.append({
        "Feature": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outlier_count,
        "Outlier %": outlier_count / len(X_train) * 100
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary.sort_values(
    "Outlier %",
    ascending=False
)

,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count,Outlier %
15,salt_intake,7.200000,9.800000,2.600000,3.300000,13.700000,26854,12.044313
14,alcohol_intake,11.900000,18.000000,6.100000,2.750000,27.150000,26112,11.711518
17,hdl,58.000000,72.000000,14.000000,37.000000,93.000000,25988,11.655902
11,homocysteine_level,12.287763,12.686175,0.398412,11.690145,13.283792,24774,11.111410
9,blood_pressure,148.946429,150.720942,1.774513,146.284659,153.382711,24303,10.900161
10,crp_level,7.294863,7.758562,0.463698,6.599316,8.454110,24155,10.833782
18,ldl,117.000000,142.000000,25.000000,79.500000,179.500000,23413,10.500987
12,systolic_bp,125.000000,144.000000,19.000000,96.500000,172.500000,21854,9.801758
13,diastolic_bp,83.000000,96.000000,13.000000,63.500000,115.500000,18579,8.332885
16,heart_rate,69.000000,80.000000,11.000000,52.500000,96.500000,16665,7.474435


In [16]:
for column in numerical_columns:
    print(f"\n--- {column} ---")

    print("Lowest values:")
    print(
        X_train[column]
        .nsmallest(5)
        .to_list()
    )

    print("Highest values:")
    print(
        X_train[column]
        .nlargest(5)
        .to_list()
    )


--- age ---
Lowest values:
[2, 2, 2, 2, 2]
Highest values:
[89, 89, 89, 89, 89]

--- age_normalized ---
Lowest values:
[0.0, 0.0, 0.0, 0.0, 0.0]
Highest values:
[1.0, 1.0, 1.0, 1.0, 1.0]

--- bmi ---
Lowest values:
[10.01, 10.01, 10.14, 10.19, 10.21]
Highest values:
[95.69, 88.76, 88.72, 87.7, 83.74]

--- hypertension ---
Lowest values:
[0.0, 0.0, 0.0, 0.0, 0.0]
Highest values:
[1.0, 1.0, 1.0, 1.0, 1.0]

--- HbA1c_level ---
Lowest values:
[3.5, 3.5, 3.5, 3.5, 3.5]
Highest values:
[9.0, 9.0, 9.0, 9.0, 9.0]

--- glucose ---
Lowest values:
[70.0, 70.0, 70.0, 70.0, 70.0]
Highest values:
[300.0, 300.0, 300.0, 300.0, 300.0]

--- cholesterol ---
Lowest values:
[150.0, 150.0, 150.0, 150.0, 150.0]
Highest values:
[300.0, 300.0, 300.0, 300.0, 300.0]

--- sleep_hours ---
Lowest values:
[4.0, 4.0, 4.0, 4.0, 4.0]
Highest values:
[10.0, 10.0, 10.0, 10.0, 10.0]

--- triglycerides ---
Lowest values:
[50.0, 50.0, 50.0, 50.0, 50.0]
Highest values:
[400.0, 400.0, 400.0, 400.0, 400.0]

--- blood_pressure

In [17]:
redundancy_columns = [
    "age",
    "age_normalized",
    "age_level",
    "bmi",
    "bmi_level"
]

X_train[
    redundancy_columns
].head(10)

,age,age_normalized,age_level,bmi,bmi_level
229021,65,0.724137,Mature,34.000000,Obese
242513,30,0.321839,Middle,17.100000,Underweight
266084,44,0.482758,Middle,27.600000,Overweight
105155,47,0.517241,Mature,33.193851,Obese
212055,27,0.287356,Middle,18.200000,Underweight
164876,32,0.344827,Middle,20.600000,Normal
28639,64,0.712644,Mature,37.410000,Obese
266004,65,0.724137,Mature,26.700000,Overweight
24983,55,0.609195,Mature,27.320000,Overweight
13083,36,0.390805,Middle,43.250000,Obese


In [18]:
print(
    "Correlation between age and age_normalized:",
    X_train["age"].corr(
        X_train["age_normalized"]
    )
)

print(
    "Number of age levels:",
    X_train["age_level"].nunique()
)

print(
    "Number of BMI levels:",
    X_train["bmi_level"].nunique()
)

Correlation between age and age_normalized: 0.9999999999991511
Number of age levels: 4
Number of BMI levels: 4


In [19]:
age_level_check = (
    X_train[
        ["age", "age_level"]
    ]
    .drop_duplicates()
    .sort_values("age")
)

age_level_check

,age,age_level
8758,2,Young
53917,3,Young
12918,4,Young
49724,5,Young
51734,6,Young
...,...,...
234881,85,Senior
205090,86,Senior
276201,87,Senior
224051,88,Senior


In [20]:
bmi_level_check = (
    X_train[
        ["bmi", "bmi_level"]
    ]
    .sort_values("bmi")
)

bmi_level_check.head(20)

,bmi,bmi_level
33933,10.01,Underweight
53998,10.01,Underweight
70982,10.14,Underweight
93543,10.19,Underweight
524,10.21,Underweight
83997,10.34,Underweight
75967,10.50,Underweight
50798,10.59,Underweight
84313,10.62,Underweight
86449,10.76,Underweight


## 3.5 Outlier Handling Decision


In [22]:
age_mapping_check = (
    X_train
    .groupby("age")["age_level"]
    .nunique()
)

bmi_mapping_check = (
    X_train
    .groupby("bmi")["bmi_level"]
    .nunique()
)

print(
    "Maximum age levels for one age value:",
    age_mapping_check.max()
)

print(
    "Maximum BMI levels for one BMI value:",
    bmi_mapping_check.max()
)

Maximum age levels for one age value: 1
Maximum BMI levels for one BMI value: 1


In [23]:
source_check = pd.DataFrame({
    "source_dataset": X_train["source_dataset"],
    classLabel: y_train
})

source_summary = (
    source_check
    .groupby("source_dataset")[classLabel]
    .agg(["count", "sum", "mean"])
)

source_summary = source_summary.rename(
    columns={
        "count": "Records",
        "sum": "Positive Cases",
        "mean": "Positive Rate"
    }
)

source_summary

,Records,Positive Cases,Positive Rate
source_dataset,,,
diabetes,75162,3135,0.041710
heart,7869,1578,0.200534
hypertension,139929,0,0.000000


## 3.6 Feature Selection

In [24]:
features_to_remove = [
    "composite_key",
    "age_normalized",
    "age_level",
    "bmi_level",
    "source_dataset"
]

X_train_selected = X_train.drop(
    columns=features_to_remove
).copy()

X_test_selected = X_test.drop(
    columns=features_to_remove
).copy()

print("Before feature selection:", X_train.shape)
print("After feature selection:", X_train_selected.shape)

print("\nRemoved features:")
print(features_to_remove)

Before feature selection: (222960, 35)
After feature selection: (222960, 30)

Removed features:
['composite_key', 'age_normalized', 'age_level', 'bmi_level', 'source_dataset']


In [25]:
categorical_columns = X_train_selected.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_cardinality = (
    X_train_selected[
        categorical_columns
    ]
    .nunique()
    .sort_values(ascending=False)
)

categorical_cardinality

smoking                 3
physical_activity       3
employment_status       3
stress_level            3
education_level         3
sugar_consumption       3
diabetes                2
gender                  2
high_ldl_cholesterol    2
low_hdl_cholesterol     2
family_history          2
high_blood_pressure     2
dtype: int64

In [26]:
from sklearn.preprocessing import OrdinalEncoder

## 3.7 Categorical Encoding

In [27]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train_processed = X_train_selected.copy()
X_test_processed = X_test_selected.copy()

X_train_processed[categorical_columns] = (
    encoder.fit_transform(
        X_train_selected[categorical_columns]
    )
)

X_test_processed[categorical_columns] = (
    encoder.transform(
        X_test_selected[categorical_columns]
    )
)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (222960, 30)
Processed testing shape: (55741, 30)


## 3.8 Numerical Variables

In [28]:
dataFeatures_train = X_train_processed.to_numpy(
    dtype=float
)

dataFeatures_test = X_test_processed.to_numpy(
    dtype=float
)

dataPhenotypes_train = y_train.to_numpy(
    dtype=int
)

dataPhenotypes_test = y_test.to_numpy(
    dtype=int
)

dataHeaders = X_train_processed.columns.to_numpy()

print("Training features:", dataFeatures_train.shape)
print("Training target:", dataPhenotypes_train.shape)

print("Testing features:", dataFeatures_test.shape)
print("Testing target:", dataPhenotypes_test.shape)

print(
    "Finite training values:",
    np.isfinite(dataFeatures_train).all()
)

print(
    "Finite testing values:",
    np.isfinite(dataFeatures_test).all()
)

Training features: (222960, 30)
Training target: (222960,)
Testing features: (55741, 30)
Testing target: (55741,)
Finite training values: True
Finite testing values: True


## 3.9 Class Imbalance

In [30]:
class_distribution = pd.DataFrame({
    "Count": y_train.value_counts(),
    "Proportion": y_train.value_counts(
        normalize=True
    )
})

class_distribution

,Count,Proportion
heart_disease_target,,
0,218247,0.978862
1,4713,0.021138


In [31]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

imbalance_ratio = (
    negative_count / positive_count
)

print(
    "Majority-to-minority ratio:",
    imbalance_ratio
)

Majority-to-minority ratio: 46.30744748567791


In [32]:
preprocessing_summary = pd.DataFrame({
    "Step": [
        "Exact duplicate removal",
        "Target leakage removal",
        "Conflicting predictor profiles",
        "Missing-value handling",
        "Outlier handling",
        "Feature selection",
        "Categorical encoding",
        "Numerical scaling",
        "Numerical discretisation",
        "Class balancing"
    ],
    "Decision": [
        "Removed exact duplicate rows",
        "Removed 4 leakage predictors",
        "Retained",
        "No missing values found",
        "Retained statistical outliers",
        "Removed 5 redundant/provenance features",
        "OrdinalEncoder fitted on training data",
        "Not required",
        "Not required",
        "Reserved for controlled Task 4 improvement"
    ]
})

preprocessing_summary

,Step,Decision
0,Exact duplicate removal,Removed exact duplicate rows
1,Target leakage removal,Removed 4 leakage predictors
2,Conflicting predictor profiles,Retained
3,Missing-value handling,No missing values found
4,Outlier handling,Retained statistical outliers
5,Feature selection,Removed 5 redundant/provenance features
6,Categorical encoding,OrdinalEncoder fitted on training data
7,Numerical scaling,Not required
8,Numerical discretisation,Not required
9,Class balancing,Reserved for controlled Task 4 improvement
